---
title: "Part 8: Pandas Core"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/08-pandas-core.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/08-pandas-core.ipynb)

**DS-MLOps Data Analysis**

**Python 3.12+ | Author: Anthony Faustine**

## Before you begin

This notebook assumes you have completed Parts 1-7 (the Python Basics section), especially the NumPy notebook (Part 4): pandas builds directly on NumPy arrays, and a lot of what looks new here is really a NumPy idea wearing a label.

Every example in this part uses a real dataset: exam results for 2,400 students across several schools, with columns covering marks, teacher counts, school size, and whether each student continued or dropped out. Real data means real messiness, missing values included, which is exactly what makes it worth learning on.

Part 9 (`09-pandas-operations.ipynb`) continues with row-wise transformations, string methods, and descriptive statistics on this same dataset.

| Topic | Why it matters |
|---|---|
| **Series and DataFrame** | The two objects every pandas operation works on |
| **Reading data** | `pd.read_csv` and the first checks every dataset deserves |
| **Selecting rows and columns** | `loc` vs `iloc`, and why mixing them up is the most common pandas bug |
| **Boolean filtering** | The same masking idea from NumPy, applied to a labelled table |
| **Missing data** | Real datasets have gaps; pandas gives you tools to see and handle them, not ignore them |

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

## Learning Objectives

By the end of Part 8 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Explain the relationship between a Series, a DataFrame, and a NumPy array | Sec. 1 |
| 2 | Load a CSV and run the first checks any new dataset deserves | Sec. 2 |
| 3 | Select columns and rows correctly with `loc` and `iloc` | Sec. 3 |
| 4 | Filter rows with boolean conditions | Sec. 4 |
| 5 | Add, modify, and drop columns without the inplace trap | Sec. 5 |
| 6 | Sort a DataFrame by one or more columns | Sec. 6 |
| 7 | Find, count, and handle missing values | Sec. 7 |

## 1. Series and DataFrame

pandas gives you two data structures, and almost everything else in the library is a method on one of them. A **Series** is a one-dimensional, labelled array: a NumPy array with an index attached. A **DataFrame** is a two-dimensional table: a collection of Series that all share the same index, one Series per column.

In [ ]:
import numpy as np
import pandas as pd

# A Series is a NumPy array plus a label for every position.
marks = pd.Series([62, 78, 91], index=["s001", "s002", "s003"])
print(marks)
print(f"underlying NumPy array: {marks.to_numpy()}")

A DataFrame is what you get when several Series that share an index sit next to each other as columns. Building one from a dict of equal-length arrays, the same pattern used to build the feature matrices in the NumPy notebook, makes this explicit:

In [ ]:
data = pd.DataFrame(
    {
        "student_id": ["s001", "s002", "s003"],
        "midterm_score": [62, 78, 91],
        "gender": ["F", "M", "F"],
    }
)
data

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: A DataFrame is a dict of same-length Series</span><br><br>
Every column of a DataFrame is a <code>Series</code> sharing the DataFrame's index. <code>data["midterm_score"]</code> returns that column as a standalone <code>Series</code>, with the same index, the same values, and every NumPy method still available on it through <code>.to_numpy()</code>. This is why the array skills from the NumPy notebook keep paying off here: a pandas column is a NumPy array with row labels attached, not a different kind of thing.
</div>

## 2. Reading and Inspecting Data

`pd.read_csv` is almost always the first line of a real analysis. The first thing to do with whatever it returns is never to start analysing it: it is to check what you actually got.

In [ ]:
df = pd.read_csv("data/university_analytics.csv")
print(f"shape   : {df.shape}")
df.head()

`.dtypes` shows what type pandas inferred for every column, `.info()` adds memory usage and non-null counts in one call, and `.describe()` summarises every numeric column at once:

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe().T

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Run these three checks on every new dataset, before anything else</span><br><br>
<code>.shape</code>, <code>.dtypes</code>, and <code>.describe()</code> take seconds and catch most of the embarrassing mistakes early: a column pandas read as text when it should be numeric, a date stored as a string, a column that is almost entirely one value. Cheaper to catch here than three analysis steps later.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - First Look</span><br><br>

<b>Goal:</b> Print the column names with <code>df.columns</code>, then print how many unique values the <code>caste</code>
column has using <code>.nunique()</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df.columns
df["program"].nunique()
# -> 4</pre>
</div>

In [ ]:
# TODO: print df.columns, then the number of unique values in "program"
...

## 3. Selecting Rows and Columns

`df["col"]` selects one column as a Series; `df[["col1", "col2"]]` selects several columns as a DataFrame, note the double brackets. Selecting rows by label or by position needs `.loc` or `.iloc`, and the two are not interchangeable.

In [ ]:
one_column = df["midterm_score"]
several_columns = df[["student_id", "midterm_score", "final_score"]]
print(type(one_column), type(several_columns))

`.loc` selects by **label**: the value in the index, or a column name. `.iloc` selects by **integer position**, exactly like a NumPy array, regardless of what the labels actually are:

In [ ]:
# .loc: by label. Row label 0, column label "midterm_score".
print(df.loc[0, "midterm_score"])

# .iloc: by position. First row, second column, whatever they are labelled.
print(df.iloc[0, 1])

# A label-based row slice with .loc is inclusive of both ends.
df.loc[0:2, ["student_id", "midterm_score"]]

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Treating <code>.loc</code> and <code>.iloc</code> as interchangeable</span><br><br>
<code>df.loc[0:2]</code> and <code>df.iloc[0:2]</code> can return a different number of rows. <code>.iloc</code> slicing behaves exactly like a Python list: the end is excluded. <code>.loc</code> slicing is inclusive of the end label, because that label might not even be an integer; it could be a date or a name. The two only happen to look similar when the index is the default <code>0, 1, 2, ...</code> range, which is exactly when this mistake hides best.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Select a Slice</span><br><br>

<b>Goal:</b> Using <code>.iloc</code>, select rows 10 through 14 (5 rows) and only the <code>student_id</code> and
<code>project_score</code> columns.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df.iloc[10:15, [?, ?]]
# Hint: find the integer position of each column with df.columns.get_loc(...)</pre>
</div>

In [ ]:
# TODO: select rows 10-14 and the student_id/project_score columns using .iloc
...

## 4. Boolean Filtering

This is the same trick from the NumPy notebook's boolean masking section, applied to a DataFrame instead of an array: a comparison produces a Series of `True`/`False`, and indexing with that Series keeps only the `True` rows.

In [ ]:
high_scorers = df["midterm_score"] > 90
print(f"high scorers: {high_scorers.sum()} of {len(df)} students")

df[high_scorers].head()

Combine conditions with `&` and `|`, the same operators and the same parentheses requirement as NumPy, not Python's `and`/`or`:

In [ ]:
struggling_without_internet = (df["final_grade"] == "F") & (~df["has_internet"])
print(f"failed students without internet: {struggling_without_internet.sum()}")

df.loc[struggling_without_internet, ["student_id", "final_grade", "has_internet"]].head()

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Female High Scorers</span><br><br>

<b>Goal:</b> Count how many female students (<code>gender == "F"</code>) scored above 80 in <code>final_score</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>condition = (df["gender"] == "F") & (df["final_score"] > 80)
condition.sum()</pre>
</div>

In [ ]:
# TODO: count female students with final_score > 80
...

## 5. Adding, Modifying, and Dropping Columns

Assigning to a new column name creates it; assigning to an existing one overwrites it. Both work the same way and both operate on the whole column at once, no loop required:

In [ ]:
df["average_marks"] = (df["midterm_score"] + df["final_score"] + df["project_score"]) / 3
df[["student_id", "average_marks"]].head()

`.drop()` removes rows or columns. `axis=1` means columns, `axis=0` (the default) means rows, the same convention as NumPy's `axis` parameter:

In [ ]:
# axis=1: drop a column. Returns a NEW DataFrame; df itself is unchanged
# unless you reassign it.
without_internet_col = df.drop("has_internet", axis=1)
print(f"original columns : {len(df.columns)}")
print(f"after drop        : {len(without_internet_col.columns)}")

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Reaching for <code>inplace=True</code> by default</span><br><br>
<code>df.drop("has_internet", axis=1, inplace=True)</code> looks like it saves memory by not creating a copy. It does not, in most pandas versions it builds the new data and discards the old either way, and it makes code harder to debug: a function that mutates its input instead of returning a new one is a common source of "why did my DataFrame change" bugs. Prefer <code>df = df.drop(...)</code>, explicit and easy to trace.
</div>

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Copy-on-Write is the default in pandas 3</span><br><br>
Pandas 3 enables <b>Copy-on-Write (CoW)</b> by default. Every operation on a subset of a DataFrame returns a new independent object instead of a view into the original. Two rules follow directly:<br><br>
<ol>
<li><b>Direct column assignment works fine:</b> <code>df["average_marks"] = ...</code> modifies <code>df</code> through <code>__setitem__</code>, not through a copy, so it still does what you expect.</li>
<li><b>Chained assignment is silently dropped:</b> <code>df[df["gender"] == "F"]["passed"] = True</code> looks like it updates <code>df</code>, but the inner <code>df[mask]</code> now returns a copy. The assignment goes to that temporary copy and is immediately lost. The fix is one step: <code>df.loc[df["gender"] == "F", "passed"] = True</code>.</li>
</ol>
CoW is why <code>inplace=True</code> (above) is being phased out — explicit reassignment (<code>df = df.drop(...)</code>) is both clearer and CoW-compatible. Part 9 shows the same rule applied to <code>groupby</code> and <code>transform</code>.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Add a Pass/Fail Column</span><br><br>

<b>Goal:</b> Add a new column <code>passed</code> that is <code>True</code> when <code>average_marks</code> is at least 0.5,
using <code>np.where</code> the same way it was used in the NumPy notebook.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df["passed"] = np.where(df["average_marks"] >= 0.5, True, False)
df["passed"].value_counts()</pre>
</div>

In [ ]:
# TODO: add the "passed" column
...

## 6. Sorting

`.sort_values()` sorts by one or more columns; `.sort_index()` sorts by the index instead. Neither changes `df` unless you reassign it, same rule as every method so far:

In [ ]:
top_5 = df.sort_values("average_marks", ascending=False).head(5)
top_5[["student_id", "average_marks"]]

In [ ]:
# Sort by multiple columns: gender first, then average_marks descending within each
by_gender_then_marks = df.sort_values(["gender", "average_marks"], ascending=[True, False])
by_gender_then_marks[["student_id", "gender", "average_marks"]].head(6)

## 7. Handling Missing Data

`total_toilets` and `establishment_year` are missing for some schools in this dataset, encoded as `NaN`. `.isna()` finds them, `.dropna()` removes rows that have them, and `.fillna()` replaces them, three different decisions with three different consequences for an analysis.

In [ ]:
missing_per_column = df.isna().sum()
print(missing_per_column[missing_per_column > 0])

Dropping every row with any missing value is the simplest option and often the wrong one: it can throw away far more data than the missing values themselves justify:

In [ ]:
print(f"rows before dropna : {len(df)}")
print(f"rows after dropna  : {len(df.dropna())}")
print(f"rows lost          : {len(df) - len(df.dropna())}")

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Decide what missing means before you decide how to handle it</span><br><br>
A missing <code>total_toilets</code> value might mean the school never reported it, which is different from meaning zero. Filling with the column mean (<code>df["total_toilets"].fillna(df["total_toilets"].mean())</code>) assumes the missing schools look like the average school. Dropping those rows assumes they are not worth analysing at all. Both are real decisions with real consequences, not a technicality to get past quickly.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 5 - Fill With the Median</span><br><br>

<b>Goal:</b> Fill the missing <code>total_toilets</code> values with the column's median instead of dropping those rows,
and confirm there are no missing values left in that column afterward.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>median_toilets = df["total_toilets"].median()
df["total_toilets"] = df["total_toilets"].fillna(median_toilets)
df["total_toilets"].isna().sum()  # -> 0</pre>
</div>

In [ ]:
# TODO: fill missing total_toilets with the column median
...

## Capstone: Top Schools Report

Combine everything from this notebook: select, filter, add a column, sort, and handle missing data, to answer one question. Which schools have the highest average student performance, among schools with reasonable internet access?

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Top Schools by Performance</span><br><br>

<b>Goal:</b>
<ol>
<li>Fill missing <code>total_toilets</code> values with the column median (Sec. 7)</li>
<li>Filter to students with <code>internet == 1</code></li>
<li>Group by <code>school_id</code> and compute the mean <code>average_marks</code> per school (a preview of Part 3's
<code>groupby</code>, used here in its simplest one-line form)</li>
<li>Sort the result descending and show the top 5 school IDs</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>with_internet = df[df["has_internet"] == 1]
school_means = with_internet.groupby("school_id")["average_marks"].mean()
school_means.sort_values(ascending=False).head(5)</pre>
</div>

In [ ]:
# TODO: build the top-5-schools report described above
...

## Further Reading

| Resource | Why it matters |
|---|---|
| McKinney, W. (2022). *Python for Data Analysis*, 3rd ed. O'Reilly. | The book by pandas' creator; free online at [wesmckinney.com/book](https://wesmckinney.com/book) — Chapters 5 and 6 map directly to this notebook |
| [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) | The official fast-start guide; read it after this notebook to see everything in one place |
| [pandas 3.0 migration guide](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html) | Covers Copy-on-Write and the new `str` dtype — the two biggest breaking changes from pandas 2 |
| Harris, C.R. et al. (2020). [Array programming with NumPy](https://doi.org/10.1038/s41586-020-2649-2). *Nature* 585, 357–362. | pandas DataFrames are built on NumPy arrays; understanding arrays makes every pandas index operation predictable |


## Summary

| Concept | Key rule |
|---|---|
| Series | A NumPy array with a label for every position |
| DataFrame | A collection of same-length Series sharing one index, one Series per column |
| `pd.read_csv` | Always follow with `.shape`, `.dtypes`, `.describe()` before analysing anything |
| `df["col"]` vs `df[["col"]]` | Single brackets return a Series, double brackets return a DataFrame |
| `.loc` | Selects by label; slices are inclusive of the end label |
| `.iloc` | Selects by integer position; slices exclude the end, like a Python list |
| Boolean filtering | `(cond1) & (cond2)`, parentheses required, same as NumPy |
| Adding columns | `df["new"] = ...` operates on the whole column, no loop needed |
| `inplace=True` | Avoid it; reassign with `df = df.method(...)` instead |
| `.sort_values()` | Pass a list of columns to sort by more than one, with a matching list for `ascending` |
| Missing data | `.isna()` to find it, `.dropna()`/`.fillna()` to handle it, decide what missing means first |

**Next:** `09-pandas-operations.ipynb`, covering row-wise transformations with `apply`, string methods, and descriptive statistics on this same dataset.